In [3]:
!pip install protobuf sentencepiece

Overwriting C:/Users/nivsa/Generation of Synthetic Training Data/embedded/extraction_engine/aligner.py


In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, 
     r"C:\Users\nivsa\Generation of Synthetic Training Data\embedded\extraction_engine\train_joint_model.py"],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

הרצת הבדיקה

In [2]:
import os
import glob
import json
from collections import defaultdict

def validate_dataset(data_dir: str):
    print(f"🔍 Starting validation on directory: {data_dir}\n")
    
    json_files = glob.glob(os.path.join(data_dir, "*.json"))
    if not json_files:
        print(f"❌ No JSON files found in {data_dir}")
        return
    
    total_samples = 0
    errors = defaultdict(list)
    
    for filepath in json_files:
        filename = os.path.basename(filepath)
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                data = json.load(f)
                
            # תמיכה גם בקובץ שהוא רשימה של דגימות וגם בדגימה בודדת
            if not isinstance(data, list):
                data = [data]
                
            for idx, sample in enumerate(data):
                total_samples += 1
                sample_id = sample.get("id", f"{filename}_sample_{idx}")
                
                # 1. בדיקה שכל המפתחות קיימים
                for key in ["tokens", "ner_tags", "relations"]:
                    if key not in sample:
                        errors["Missing Keys"].append(f"{sample_id}: missing '{key}'")
                
                if "Missing Keys" in errors and errors["Missing Keys"][-1].startswith(sample_id):
                    continue # דילוג על דגימה חסרה לחלוטין
                        
                tokens = sample["tokens"]
                ner_tags = sample["ner_tags"]
                relations = sample["relations"]
                
                # 2. חוק האורך: מספר הטוקנים חייב להיות זהה למספר התגיות
                if len(tokens) != len(ner_tags):
                    errors["Length Mismatch"].append(f"{sample_id}: {len(tokens)} tokens vs {len(ner_tags)} tags")
                    
                # 3. חוקי BIO: תגית I חייבת לבוא אחרי B או I מאותו סוג
                for i, tag in enumerate(ner_tags):
                    if tag.startswith("I-"):
                        base_tag = tag[2:]
                        if i == 0:
                            errors["Invalid BIO"].append(f"{sample_id}: Starts with I- tag at index 0 ('{tokens[i]}')")
                        else:
                            prev_tag = ner_tags[i-1]
                            if prev_tag == "O" or prev_tag[2:] != base_tag:
                                errors["Invalid BIO"].append(f"{sample_id}: I-{base_tag} follows {prev_tag} at index {i} ('{tokens[i]}')")
                                
                # 4. חוקי קשרים (Relations)
                for r_idx, rel in enumerate(relations):
                    head = rel.get("head", [])
                    tail = rel.get("tail", [])
                    
                    if not head or not tail:
                        errors["Empty Relation"].append(f"{sample_id}: Relation {r_idx} has empty head or tail")
                        continue
                        
                    # א. חריגה מגבולות המערך
                    if max(head) >= len(tokens) or max(tail) >= len(tokens):
                        errors["Relation Out of Bounds"].append(f"{sample_id}: Relation {r_idx} indices exceed token length")
                        continue
                        
                    # ב. בדיקה לוגית: האם ה-Head בכלל מצביע לישות? (הוא לא אמור להיות "O")
                    head_start_tag = ner_tags[head[0]]
                    if head_start_tag == "O":
                        errors["Relation to Non-Entity"].append(f"{sample_id}: Relation head points to 'O' at index {head[0]} ('{tokens[head[0]]}')")

        except Exception as e:
            errors["File Parsing Error"].append(f"{filename}: {str(e)}")

    # ---------------------------------------------------------
    # דוח סופי
    # ---------------------------------------------------------

    print("=" * 40)
    print("📊 VALIDATION REPORT")
    print("=" * 40)
    print(f"Files processed:   {len(json_files):,}")
    print(f"Samples checked:   {total_samples:,}")
    print("=" * 40 + "\n")
    
    if not errors:
        print("✅ PERFECT! 0 errors found.")
        print("Your dataset is 100% clean, consistent, and ready for production training! 🚀")
    else:
        print("🚨 ERRORS FOUND:")
        for err_type, err_list in errors.items():
            print(f"\n[{err_type}] - {len(err_list)} issues found:")
            # נדפיס רק את ה-5 הראשונים מכל סוג כדי לא להציף את המסך
            for e in err_list[:5]:
                print(f"  - {e}")
            if len(err_list) > 5:
                print(f"  ... and {len(err_list) - 5} more.")

if __name__ == '__main__':
    # זו הכתובת שכבר השתמשנו בה, תוודא רק שהיא עדיין מעודכנת אצלך:
    DATA_DIR = r"C:\Users\nivsa\Generation of Synthetic Training Data\embedded\final_dataset\json_output"
    
    validate_dataset(DATA_DIR)

🔍 Starting validation on directory: C:\Users\nivsa\Generation of Synthetic Training Data\embedded\final_dataset\json_output

📊 VALIDATION REPORT
Files processed:   7,991
Samples checked:   11,862

✅ PERFECT! 0 errors found.
Your dataset is 100% clean, consistent, and ready for production training! 🚀


Overwriting C:/Users/nivsa/Generation of Synthetic Training Data/embedded/extraction_engine/train_joint_model.py


In [12]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, 
     r"C:\Users\nivsa\Generation of Synthetic Training Data\embedded\extraction_engine\train_joint_model.py"],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

Using device: cpu
Loaded 35 samples from 22 files.
Sanity Check — Train: 28 | Val: 7

STDERR: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
Traceback (most recent call last):
  File "C:\Users\nivsa\Generation of Synthetic Training Data\embedded\extraction_engine\train_joint_model.py", line 473, in <module>
    main()
  File "C:\Users\nivsa\Generation of Synthetic Training Data\embedded\extraction_engine\train_joint_model.py", line 438, in main
    print(f"Val set saved: {len(val_samples)} samples \u2192 {val_save_path}")
  File "C:\Users\nivsa\Anaconda3\envs\ollama-env\lib\encodings\cp1255.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
UnicodeEncodeError: 'charmap' codec can't encode character '\u2192' in position 25: character maps to <undefined>



בדיקת התוצאות

In [ ]:
# ----------------------------------------------------------------------------
# Donut Inference for Jupyter Notebook
# ----------------------------------------------------------------------------

# --- Imports ---
import json
import os
import re
import torch
from pathlib import Path
from PIL import Image
from transformers import VisionEncoderDecoderModel, DonutProcessor
import matplotlib.pyplot as plt

# ============================================================================
# CONFIGURATION (Change these paths!)
# ============================================================================

# Path to your fine-tuned model folder (where train.py saved it)
MODEL_PATH = "./donut-datasheets-finetuned"

# Path to the image you want to test
IMAGE_PATH = r"C:\Users\nivsa\Generation of Synthetic Training Data\embedded\final_dataset\images\datasheet_00eb2cd6.jpg"

# Device (cuda or cpu)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ============================================================================
# FUNCTIONS
# ============================================================================

def load_model(model_path, device="cuda"):
    print(f"Loading model from: {model_path}")
    try:
        processor = DonutProcessor.from_pretrained(model_path)
        model = VisionEncoderDecoderModel.from_pretrained(model_path)
    except OSError as e:
        print(f"[X] Error: Could not load model from {model_path}")
        print(f"  Details: {e}")
        return None, None

    model.to(device)
    model.eval()
    
    print(f"[OK] Model loaded on {device}")
    print(f"[OK] Input Resolution: {processor.image_processor.size}")
    return model, processor

def extract_json(image_path, model, processor, device="cuda"):
    print(f"\nProcessing: {os.path.basename(image_path)}")
    
    try:
        image = Image.open(image_path).convert("RGB")
    except Exception as e:
        return {"error": str(e)}

    # Display image in notebook
    plt.figure(figsize=(10, 10))
    plt.imshow(image)
    plt.axis('off')
    plt.show()

    # Prepare input
    pixel_values = processor(image, return_tensors="pt").pixel_values.to(device)
    task_prompt = "<s_datasheet>"
    decoder_input_ids = processor.tokenizer(task_prompt, add_special_tokens=False, return_tensors="pt").input_ids.to(device)

    # Generate
    print("Generating output...")
    with torch.no_grad():
        outputs = model.generate(
            pixel_values,
            decoder_input_ids=decoder_input_ids,
            max_length=1024,
            early_stopping=True,
            pad_token_id=processor.tokenizer.pad_token_id,
            eos_token_id=processor.tokenizer.eos_token_id,
            use_cache=True,
            return_dict_in_generate=True,
        )

    # Decode
    seq = processor.tokenizer.batch_decode(outputs.sequences, skip_special_tokens=True)[0]
    clean_seq = seq.replace(task_prompt, "").strip()
    
    # Try parsing JSON
    try:
        return json.loads(clean_seq)
    except json.JSONDecodeError:
        # Fallback: Try to find JSON object in text
        match = re.search(r'\{.*\}', clean_seq, re.DOTALL)
        if match:
            try:
                return json.loads(match.group(0))
            except:
                pass
        
        # Fallback: Fix missing braces
        if clean_seq.startswith("{") and not clean_seq.endswith("}"):
            try:
                return json.loads(clean_seq + "}")
            except:
                pass
                
        return {"raw_output": clean_seq, "error": "Failed to parse JSON"}

# ============================================================================
# MAIN EXECUTION
# ============================================================================

# 1. Load Model (Only runs once if you keep the cell output)
if 'model' not in globals():
    model, processor = load_model(MODEL_PATH, DEVICE)

# 2. Run Inference
if model:
    result = extract_json(IMAGE_PATH, model, processor, DEVICE)
    
    print("\n" + "="*60)
    print("EXTRACTION RESULT")
    print("="*60)
    print(json.dumps(result, indent=2, ensure_ascii=False))

In [3]:
import json
import os

# שים פה את הנתיב לתיקייה שמכילה את כל קבצי ה-JSON שעברו Preprocessing
output_dir = r"C:\Users\nivsa\Generation of Synthetic Training Data\embedded\final_dataset\json_output"

total_windows = 0
total_tokens = 0
total_relations = 0
total_entities = 0

for filename in os.listdir(output_dir):
    if filename.endswith(".json"):
        with open(os.path.join(output_dir, filename), 'r', encoding='utf-8') as f:
            data = json.load(f)
            
            # הקובץ מכיל רשימה של חלונות (Sliding Windows) למסמך
            for sample in data:
                total_windows += 1
                
                # ספירת טוקנים
                total_tokens += len(sample.get("tokens", []))
                
                # ספירת קשרים (Relations)
                total_relations += len(sample.get("relations", []))
                
                # ספירת ישויות: כל ישות מוגדרת על ידי תחילת תגית (B-)
                ner_tags = sample.get("ner_tags", [])
                entities_in_sample = sum(1 for tag in ner_tags if tag.startswith("B-"))
                total_entities += entities_in_sample

print("=== 📊 Dataset Statistics ===")
print(f"Total Documents / Sliding Windows: {total_windows:,}")
print(f"Avg Tokens per Window: {total_tokens / total_windows:.0f}" if total_windows > 0 else 0)
print(f"Total Annotated Entities: {total_entities:,}")
print(f"Total Semantic Relations: {total_relations:,}")

=== 📊 Dataset Statistics ===
Total Documents / Sliding Windows: 11,862
Avg Tokens per Window: 379
Total Annotated Entities: 518,824
Total Semantic Relations: 195,515


Writing C:/Users/nivsa/Generation of Synthetic Training Data/embedded/extraction_engine/evaluate.py


In [19]:
import subprocess, sys

base       = r"C:\Users\nivsa\Generation of Synthetic Training Data\embedded"
script_dir = rf"{base}\extraction_engine"
model_dir  = rf"{base}\models"
data_dir   = rf"{base}\data"

result = subprocess.run(
    [sys.executable, rf"{script_dir}\evaluate.py",
     "--model_path", rf"{model_dir}\checkpoints\checkpoint_epoch_3.pt",
     "--val_path",   rf"{data_dir}\splits\val.json",
     "--output_dir", rf"{model_dir}\eval_results"],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)


STDERR: Traceback (most recent call last):
  File "C:\Users\nivsa\Generation of Synthetic Training Data\embedded\extraction_engine\evaluate.py", line 22, in <module>
    from seqeval.metrics import classification_report as seq_classification_report
ModuleNotFoundError: No module named 'seqeval'



In [14]:
import os

base = r"C:\Users\nivsa\Generation of Synthetic Training Data\embedded"

moves = [
    # (מקור, יעד)
    # data/raw
    (r"final_dataset\html",                          r"data\raw\html"),
    (r"final_dataset\production_20260222_110334.jsonl", r"data\raw\production_20260222.jsonl"),
    (r"final_dataset\production_20260226_093915.jsonl", r"data\raw\production_20260226.jsonl"),

    # data/processed
    (r"final_dataset\sanity_check_json",             r"data\processed"),

    # data/splits
    (r"final_dataset\model tests\val.json",          r"data\splits\val.json"),

    # models
    (r"final_dataset\model tests\checkpoint_epoch_1.pt", r"models\checkpoints\checkpoint_epoch_1.pt"),
    (r"final_dataset\model tests\checkpoint_epoch_2.pt", r"models\checkpoints\checkpoint_epoch_2.pt"),
    (r"final_dataset\model tests\checkpoint_epoch_3.pt", r"models\checkpoints\checkpoint_epoch_3.pt"),

    # assets
    (r"Gemini_Generated_Image_h0alkhh0alkhh0al.png", r"assets\Gemini_Generated_Image_h0alkhh0alkhh0al.png"),
    (r"Gemini_Generated_Image_kr0z7nkr0z7nkr0z.png", r"assets\Gemini_Generated_Image_kr0z7nkr0z7nkr0z.png"),
    (r"Preprocessing _Pipeline.png",                 r"assets\Preprocessing_Pipeline.png"),
    (r"Project Overview Workflow and Methodology for Datasheet Extraction.pptx", r"assets\Project_Overview.pptx"),

    # notebooks
    (r"Synthetic_Datasheet_Pipeline.ipynb",          r"notebooks\01_generation.ipynb"),
    (r"Donut_Datasheet_FineTuning.ipynb",            r"notebooks\02_training.ipynb"),
]

print("הקבצים הבאים יועברו:\n")
for src, dst in moves:
    src_full = os.path.join(base, src)
    dst_full = os.path.join(base, dst)
    exists   = "✅" if os.path.exists(src_full) else "❌ לא קיים"
    print(f"  {exists}  {src}")
    print(f"       → {dst}\n")

הקבצים הבאים יועברו:

  ✅  final_dataset\html
       → data\raw\html

  ✅  final_dataset\production_20260222_110334.jsonl
       → data\raw\production_20260222.jsonl

  ✅  final_dataset\production_20260226_093915.jsonl
       → data\raw\production_20260226.jsonl

  ✅  final_dataset\sanity_check_json
       → data\processed

  ✅  final_dataset\model tests\val.json
       → data\splits\val.json

  ✅  final_dataset\model tests\checkpoint_epoch_1.pt
       → models\checkpoints\checkpoint_epoch_1.pt

  ✅  final_dataset\model tests\checkpoint_epoch_2.pt
       → models\checkpoints\checkpoint_epoch_2.pt

  ✅  final_dataset\model tests\checkpoint_epoch_3.pt
       → models\checkpoints\checkpoint_epoch_3.pt

  ✅  Gemini_Generated_Image_h0alkhh0alkhh0al.png
       → assets\Gemini_Generated_Image_h0alkhh0alkhh0al.png

  ✅  Gemini_Generated_Image_kr0z7nkr0z7nkr0z.png
       → assets\Gemini_Generated_Image_kr0z7nkr0z7nkr0z.png

  ✅  Preprocessing _Pipeline.png
       → assets\Preprocessing_Pipeline

In [15]:
import os
import shutil

base = r"C:\Users\nivsa\Generation of Synthetic Training Data\embedded"

moves = [
    (r"final_dataset\html",                                                        r"data\raw\html"),
    (r"final_dataset\production_20260222_110334.jsonl",                            r"data\raw\production_20260222.jsonl"),
    (r"final_dataset\production_20260226_093915.jsonl",                            r"data\raw\production_20260226.jsonl"),
    (r"final_dataset\sanity_check_json",                                           r"data\processed"),
    (r"final_dataset\model tests\val.json",                                        r"data\splits\val.json"),
    (r"final_dataset\model tests\checkpoint_epoch_1.pt",                           r"models\checkpoints\checkpoint_epoch_1.pt"),
    (r"final_dataset\model tests\checkpoint_epoch_2.pt",                           r"models\checkpoints\checkpoint_epoch_2.pt"),
    (r"final_dataset\model tests\checkpoint_epoch_3.pt",                           r"models\checkpoints\checkpoint_epoch_3.pt"),
    (r"Gemini_Generated_Image_h0alkhh0alkhh0al.png",                              r"assets\Gemini_Generated_Image_h0alkhh0alkhh0al.png"),
    (r"Gemini_Generated_Image_kr0z7nkr0z7nkr0z.png",                              r"assets\Gemini_Generated_Image_kr0z7nkr0z7nkr0z.png"),
    (r"Preprocessing _Pipeline.png",                                               r"assets\Preprocessing_Pipeline.png"),
    (r"Project Overview Workflow and Methodology for Datasheet Extraction.pptx",   r"assets\Project_Overview.pptx"),
    (r"Synthetic_Datasheet_Pipeline.ipynb",                                        r"notebooks\01_generation.ipynb"),
    (r"Donut_Datasheet_FineTuning.ipynb",                                          r"notebooks\02_training.ipynb"),
]

for src, dst in moves:
    src_full = os.path.join(base, src)
    dst_full = os.path.join(base, dst)
    os.makedirs(os.path.dirname(dst_full), exist_ok=True)
    shutil.move(src_full, dst_full)
    print(f"✅ הועבר: {src} → {dst}")

print("\nהעברה הושלמה.")

✅ הועבר: final_dataset\html → data\raw\html
✅ הועבר: final_dataset\production_20260222_110334.jsonl → data\raw\production_20260222.jsonl
✅ הועבר: final_dataset\production_20260226_093915.jsonl → data\raw\production_20260226.jsonl
✅ הועבר: final_dataset\sanity_check_json → data\processed
✅ הועבר: final_dataset\model tests\val.json → data\splits\val.json
✅ הועבר: final_dataset\model tests\checkpoint_epoch_1.pt → models\checkpoints\checkpoint_epoch_1.pt
✅ הועבר: final_dataset\model tests\checkpoint_epoch_2.pt → models\checkpoints\checkpoint_epoch_2.pt
✅ הועבר: final_dataset\model tests\checkpoint_epoch_3.pt → models\checkpoints\checkpoint_epoch_3.pt
✅ הועבר: Gemini_Generated_Image_h0alkhh0alkhh0al.png → assets\Gemini_Generated_Image_h0alkhh0alkhh0al.png
✅ הועבר: Gemini_Generated_Image_kr0z7nkr0z7nkr0z.png → assets\Gemini_Generated_Image_kr0z7nkr0z7nkr0z.png
✅ הועבר: Preprocessing _Pipeline.png → assets\Preprocessing_Pipeline.png
✅ הועבר: Project Overview Workflow and Methodology for Datas

In [18]:
path = r"C:\Users\nivsa\Generation of Synthetic Training Data\embedded\extraction_engine\evaluate.py"

with open(path, "r", encoding="utf-8") as f:
    content = f.read()

replacements = {
    r"C:\Users\nivsa\Generation of Synthetic Training Data\embedded\final_dataset\model tests\best_model.pt":
        r"C:\Users\nivsa\Generation of Synthetic Training Data\embedded\models\checkpoints\checkpoint_epoch_3.pt",

    r"C:\Users\nivsa\Generation of Synthetic Training Data\embedded\final_dataset\model tests\val.json":
        r"C:\Users\nivsa\Generation of Synthetic Training Data\embedded\data\splits\val.json",

    r"C:\Users\nivsa\Generation of Synthetic Training Data\embedded\final_dataset\model tests\eval_results":
        r"C:\Users\nivsa\Generation of Synthetic Training Data\embedded\models\eval_results",
}

for old, new in replacements.items():
    content = content.replace(old, new)

with open(path, "w", encoding="utf-8") as f:
    f.write(content)

print("✅ evaluate.py עודכן")

✅ evaluate.py עודכן


In [24]:
import os

base = r"C:\Users\nivsa\Generation of Synthetic Training Data\embedded"

# כתובות ישנות שלא אמורות להופיע יותר
old_paths = [
    r"final_dataset\html",
    r"final_dataset\sanity_check_json",
    r"final_dataset\model tests",
    r"final_dataset\json_output",
    r"final_dataset\checkpoints_sanity",
]

# קבצים לבדוק
files_to_check = [
    r"extraction_engine\aligner.py",
    r"extraction_engine\train_joint_model.py",
    r"extraction_engine\evaluate.py",
]

print("=" * 60)
any_issue = False

for rel_file in files_to_check:
    filepath = os.path.join(base, rel_file)
    print(f"\n📄 {rel_file}")

    if not os.path.exists(filepath):
        print(f"  ❌ קובץ לא נמצא")
        any_issue = True
        continue

    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read()

    file_clean = True
    for old in old_paths:
        if old in content:
            print(f"  ⚠️  עדיין מכיל: {old}")
            any_issue = True
            file_clean = False

    if file_clean:
        print(f"  ✅ נקי")

print("\n" + "=" * 60)
if any_issue:
    print("⚠️  יש כתובות שצריך לעדכן")
else:
    print("✅ כל הכתובות עודכנו בהצלחה")


📄 extraction_engine\aligner.py
  ✅ נקי

📄 extraction_engine\train_joint_model.py
  ✅ נקי

📄 extraction_engine\evaluate.py
  ✅ נקי

✅ כל הכתובות עודכנו בהצלחה
